# DEFINE OUR CONSTANTS THROUGHT HTIS WORK

In [3]:
WHO_DAILY=15 #ug/m3 , who 2021 hour guideline

In [14]:
import pandas as pd
df=pd.read_csv('data/ACCRA_ALL_SITES_PM25_HOURLY.csv',low_memory=False)
#the date is a string let's parse into datetime
df['date']=pd.to_datetime(df['datetime_utc'],'coerce')
#set the date column as index




In [12]:
df.head()

,location_id,location_name,locality,sensor_id,datetime_utc,datetime_local,datetime_to_utc,datetime_to_local,pm25_value,latitude,longitude,provider,timezone,date
0,6313993,"+233, North Ridge",NaN,15998899,2026-04-19 20:00:00+00:00,2026-04-19T20:00:00Z,2026-04-19T21:00:00Z,2026-04-19T21:00:00Z,11.60,5.57375,-0.19544,Clarity,Africa/Accra,2026-04-19 20:00:00+00:00
1,6313993,"+233, North Ridge",NaN,15998899,2026-04-21 18:00:00+00:00,2026-04-21T18:00:00Z,2026-04-21T19:00:00Z,2026-04-21T19:00:00Z,5.17,5.57375,-0.19544,Clarity,Africa/Accra,2026-04-21 18:00:00+00:00
2,6313993,"+233, North Ridge",NaN,15998899,2026-04-21 21:00:00+00:00,2026-04-21T21:00:00Z,2026-04-21T22:00:00Z,2026-04-21T22:00:00Z,15.20,5.57375,-0.19544,Clarity,Africa/Accra,2026-04-21 21:00:00+00:00
3,6313993,"+233, North Ridge",NaN,15998899,2026-04-22 00:00:00+00:00,2026-04-22T00:00:00Z,2026-04-22T01:00:00Z,2026-04-22T01:00:00Z,40.90,5.57375,-0.19544,Clarity,Africa/Accra,2026-04-22 00:00:00+00:00
4,6313993,"+233, North Ridge",NaN,15998899,2026-04-22 01:00:00+00:00,2026-04-22T01:00:00Z,2026-04-22T02:00:00Z,2026-04-22T02:00:00Z,21.30,5.57375,-0.19544,Clarity,Africa/Accra,2026-04-22 01:00:00+00:00


In [ ]:
#coerce pm25 values to numeric, errors will be set to NaN
df['pm25_value']=pd.to_numeric(df['pm25_value'],errors='coerce')
#let's check duplicate sensor readings, we will drop duplicates
df.duplicated(subset=['sensor_id','date']).sum()


np.int64(0)

Let's do a completeness check get a table considering the expected, the observed as well as the first and last readings, we'll use just one of th emost populated ones for our analyssis, you can use any for yours

In [37]:
summary=df.groupby(['sensor_id','location_name']).agg(first=('date','min'),last=('date','max'),observed=('pm25_value','count'),
                                              expected=('date',lambda x: (x.max()-x.min()).total_seconds()/3600+1)).sort_values('observed',ascending=False).head(10)

In [ ]:
#let's see the 20 worse sites in terms of completeness of data, we will use the expected and observed columns to calculate completeness
summary['completness']=summary['observed']/summary['expected']*100
summary.sort_values('completness').head(20)

,,first,last,observed,expected,completness
sensor_id,location_name,,,,,
10826686,Amasaman,2025-01-01 00:00:00+00:00,2026-09-29 12:00:00+00:00,10803,15277.0,70.714145
6530278,Physics Department-UG-Accra/CAOA-UESD,2025-01-01 00:00:00+00:00,2026-09-29 12:00:00+00:00,11293,15277.0,73.921581
10330996,37 Lorry Station,2025-01-01 00:00:00+00:00,2026-09-29 12:00:00+00:00,11707,15277.0,76.631538
10331000,Kwashieman Presby Church,2025-01-01 00:00:00+00:00,2026-09-29 13:00:00+00:00,12124,15278.0,79.355937
10331005,Lapaz Intersection,2025-01-01 00:00:00+00:00,2026-09-29 13:00:00+00:00,12256,15278.0,80.219924
10330999,West Hills Mall,2025-01-01 00:00:00+00:00,2026-09-29 13:00:00+00:00,12275,15278.0,80.344286
10330997,Ashaley Botwe School Junction,2025-01-01 00:00:00+00:00,2026-09-29 13:00:00+00:00,12276,15278.0,80.350831
7897442,Afri-SET,2025-01-01 00:00:00+00:00,2026-08-12 16:00:00+00:00,12248,14129.0,86.686956
7897459,Afri-SET,2025-01-01 00:00:00+00:00,2026-08-12 16:00:00+00:00,12258,14129.0,86.757732


We can't really asess all the sites but let's say lapaz intersection is the site of interest, let's pick it find out concerningduplicates, dsort it for time series and rebuild a gap free hourly analysis

In [62]:
#let's strip off string of th elocation names so that spaces don't cause issues when we filter by location name
df['location_name']=df['location_name'].str.strip()
#first let's sort, drop duplicate timesteps and index by date
lapaz=df[df['location_name']=='Lapaz Intersection'].sort_values('date').drop_duplicates(subset='date').set_index('date')

In [64]:
lapaz['pm25_value'].isna().sum()

np.int64(0)

We know that lapaz isn't complete, so we gotta time interpolate but we are careful to fill missing runs of 1-4 consecutive hours, longer gaps must stay nan

In [ ]:
#let's find the gaps in the data
lapaz['gap']=lapaz.index.to_series().diff().dt.total_seconds()/3600
lapaz['gap'].value_counts().sort_index()
#i am specifically trying to fill gaps greater than one hour , that means to to give them a new pm25_value as a rolling mean
#first resample so that we insert gaps betweeen the dats, the orignal pm_25 has no na, so pandas doesn't know how to insert
lapaz_full=lapaz.resample('1H').asfreq()
rolling_mean=lapaz['pm25_value'].rolling(4,min_periods=1).mean()
is_na=lapaz_full['pm25_value'].isna()
gap_groups=(~is_na).cumsum()[is_na]

gap
1.0     10956
2.0       635
3.0       303
4.0       138
5.0        86
6.0        40
7.0        35
8.0        22
9.0        12
10.0        5
11.0        6
12.0        4
13.0        3
14.0        3
17.0        2
19.0        1
21.0        1
22.0        2
28.0        1
Name: count, dtype: int64

In [61]:
#print expected , hours filled and observed hours
#i can do expected hours as the difference between the last and first date in hours,observed hours as the count of non null pm25 values and filled hours as the count of non null pm25_value_filled values
lapaz_summary=pd.DataFrame({'expected_hours':(lapaz.index.max()-lapaz.index.min()).total_seconds()/3600,
                            'observed_hours':lapaz['pm25_value'].count(),'filled_hours':lapaz['pm25_value_filled'].count(),},index=lapaz)
lapaz_summary

,expected_hours,observed_hours,filled_hours
"(3025586, Lapaz Intersection, nan, 10331005, 2025-01-01 00:00:00+00:00, 2025-01-01T00:00:00Z, 2025-01-01T01:00:00Z, 2025-01-01T01:00:00Z, 87.6, 5.60724, -0.24896, Clarity, Africa/Accra, nan, 87.6)",15277.0,12256,12256
"(3025586, Lapaz Intersection, nan, 10331005, 2025-01-01 01:00:00+00:00, 2025-01-01T01:00:00Z, 2025-01-01T02:00:00Z, 2025-01-01T02:00:00Z, 84.2, 5.60724, -0.24896, Clarity, Africa/Accra, 1.0, 84.2)",15277.0,12256,12256
"(3025586, Lapaz Intersection, nan, 10331005, 2025-01-01 02:00:00+00:00, 2025-01-01T02:00:00Z, 2025-01-01T03:00:00Z, 2025-01-01T03:00:00Z, 92.9, 5.60724, -0.24896, Clarity, Africa/Accra, 1.0, 92.9)",15277.0,12256,12256
"(3025586, Lapaz Intersection, nan, 10331005, 2025-01-01 03:00:00+00:00, 2025-01-01T03:00:00Z, 2025-01-01T04:00:00Z, 2025-01-01T04:00:00Z, 74.2, 5.60724, -0.24896, Clarity, Africa/Accra, 1.0, 74.2)",15277.0,12256,12256
"(3025586, Lapaz Intersection, nan, 10331005, 2025-01-01 04:00:00+00:00, 2025-01-01T04:00:00Z, 2025-01-01T05:00:00Z, 2025-01-01T05:00:00Z, 55.4, 5.60724, -0.24896, Clarity, Africa/Accra, 1.0, 55.4)",15277.0,12256,12256
...,...,...,...
"(3025586, Lapaz Intersection, nan, 10331005, 2026-09-29 09:00:00+00:00, 2026-09-29T09:00:00Z, 2026-09-29T10:00:00Z, 2026-09-29T10:00:00Z, 25.4, 5.60724, -0.24896, Clarity, Africa/Accra, 1.0, 25.4)",15277.0,12256,12256
"(3025586, Lapaz Intersection, nan, 10331005, 2026-09-29 10:00:00+00:00, 2026-09-29T10:00:00Z, 2026-09-29T11:00:00Z, 2026-09-29T11:00:00Z, 23.7, 5.60724, -0.24896, Clarity, Africa/Accra, 1.0, 23.7)",15277.0,12256,12256
"(3025586, Lapaz Intersection, nan, 10331005, 2026-09-29 11:00:00+00:00, 2026-09-29T11:00:00Z, 2026-09-29T12:00:00Z, 2026-09-29T12:00:00Z, 21.1, 5.60724, -0.24896, Clarity, Africa/Accra, 1.0, 21.1)",15277.0,12256,12256
"(3025586, Lapaz Intersection, nan, 10331005, 2026-09-29 12:00:00+00:00, 2026-09-29T12:00:00Z, 2026-09-29T13:00:00Z, 2026-09-29T13:00:00Z, 13.9, 5.60724, -0.24896, Clarity, Africa/Accra, 1.0, 13.9)",15277.0,12256,12256


In [ ]:
lapaz['gap'].value_counts().sort_index()
lapaz['pm25_value_filled']=lapaz['pm25_value'].fillna(lapaz['pm25_value'].rolling(4,min_periods=1).mean())